# Patch-level labels from Segmented ImageNet

This notebook batches images and queries labels for individual spatial patches in the [Segmented ImageNet-1K Subset](https://huggingface.co/datasets/Prisma-Multimodal/segmented-imagenet1k-subset). Install this checkout with `pip install -e .` from the repository root and open the notebook with that Python environment.

Set `data_dir` to the folder containing `test.json` and the extracted `images/` and `masks/` directories. If you do not have the files, set `download_data = True` to let Hugging Face Datasets download and cache them. The image and mask archives contain all splits and require about 8.5 GB of downloads.

In [ ]:
import os
from pathlib import Path

from torch.utils.data import DataLoader

from vit_prisma.dataloaders.patch_labels import PatchLabelDataset, collate_patch_labels

data_dir = os.environ.get("SEGMENTED_IMAGENET_DIR")  # Or set the folder path here.
download_data = False
split = "test"  # Also accepts "train" or "val".
patch_size = 16  # Try 8 or 32 for a different patch grid.

if data_dir:
    dataset = PatchLabelDataset.from_json(Path(data_dir) / f"{split}.json", patch_size=patch_size)
elif download_data:
    dataset = PatchLabelDataset.from_huggingface(split, patch_size=patch_size)
else:
    raise ValueError("Set data_dir or enable download_data before running this cell.")

In [ ]:
loader = DataLoader(dataset, batch_size=2, collate_fn=collate_patch_labels)
images, batch_labels = next(iter(loader))
print(f"Image batch shape: {tuple(images.shape)}")

Patch numbers start at zero in the upper-left and advance across each row. They index spatial patches only. A patch can have multiple overlapping labels, and an unlabeled patch returns an empty list.

In [ ]:
rows, cols = batch_labels[0].grid_size
patch_number = (rows // 2) * cols + (cols // 2)
for image_index, patch_labels in enumerate(batch_labels):
    print(f"Image {image_index}, patch {patch_number}: {patch_labels.labels_for_patch(patch_number)}")